# ML Tutor — Week 6: Tuning, fairness & explainability
**Track:** Core ML · **Lab:** Lab 6 — Tune, audit, and explain an equitable recommendation model

**Objectives this week:**
1. Tune models with grid/random search and read validation curves. *(CO5)*
2. Audit performance across patient subgroups and read SHAP/LIME explanations. *(CO6)*
3. Mitigate an observed fairness gap and communicate model drivers. *(CO5, CO6)*

> Anti-shortcut rule: hints live in ML Tutor, revealed one tier at a time. Work here, check hints there. You are graded on unaided competence.


**Dataset:** Synthetic treatment-support dataset with columns such as age_band, comorbidity_count, prior_utilization, adherence_score, access_barrier_flag, subgroup, and recommended_support (binary target). All records are fictional and for educational model-evaluation practice only.


## 1. Build a safe baseline pipeline
Create a train/test split and a pipeline with preprocessing plus a baseline decision tree.

**Checkpoint:** Checkpoint 1 verifies that the train/test split exists, preprocessing is inside the pipeline, and the baseline model is fit without touching the test labels during training.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

# df is preloaded in the notebook.
target = "recommended_support"
feature_cols = [
    "age_band",
    "comorbidity_count",
    "prior_utilization",
    "adherence_score",
    "access_barrier_flag",
    "subgroup",
]

X = df[feature_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=7,
)

numeric_features = ["comorbidity_count", "prior_utilization", "adherence_score"]
categorical_features = ["age_band", "access_barrier_flag", "subgroup"]

# TODO: build a preprocessing transformer for numeric + categorical features
preprocess = ...

baseline = Pipeline(
    steps=[
        ("prep", preprocess),
        ("model", DecisionTreeClassifier(random_state=7)),
    ]
)

# TODO: fit baseline on the TRAIN split only



## 2. Tune one model honestly
Run a small grid search over tree depth and leaf size using recall as the scoring metric, then compare tuned versus baseline results.

**Checkpoint:** Checkpoint 2 verifies that GridSearchCV ran on the training split, used a valid scoring string, produced a best-parameter dictionary plus mean CV score, and included a short tuned-vs-baseline comparison note.


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__max_depth": [2, 4, 6, None],
    "model__min_samples_leaf": [1, 3, 6],
}

search = GridSearchCV(
    estimator=baseline,
    param_grid=param_grid,
    scoring=...,      # TODO: choose the metric we care about today
    cv=5,
    n_jobs=None,
)

# TODO: fit search on X_train, y_train

print("Best params:", ...)
print("Best mean CV score:", ...)

# TODO: compare the tuned mean CV recall against your untuned baseline.
# In one comment, write whether the improvement is meaningful or tiny.



## 3. Audit subgroup performance
Compare recall and false-negative counts across subgroups on held-out predictions, then decide whether the gap is operationally important.

**Checkpoint:** Checkpoint 3 verifies that students produced a table with subgroup, recall, false_negatives, and n, identified which subgroup had the lowest recall on the held-out set, and wrote a brief judgment note rather than only printing numbers.


In [ ]:
from sklearn.metrics import recall_score, confusion_matrix
import pandas as pd

best_model = ...   # TODO: recover the tuned estimator
y_pred = ...       # TODO: predict on X_test

audit = X_test[["subgroup"]].copy()
audit["y_true"] = y_test.to_numpy()
audit["y_pred"] = y_pred

rows = []
for subgroup_name, group_df in audit.groupby("subgroup"):
    # TODO: compute recall for this subgroup
    subgroup_recall = ...

    # TODO: count false negatives from y_true / y_pred
    false_negatives = ...

    rows.append(
        {
            "subgroup": subgroup_name,
            "recall": subgroup_recall,
            "false_negatives": false_negatives,
            "n": len(group_df),
        }
    )

audit_table = pd.DataFrame(rows).sort_values("recall")
print(audit_table)

# TODO: write 2-3 sentences:
# - Which subgroup had the weakest recall?
# - Is the gap large enough to worry you in this course scenario?
# - What extra evidence would you want before changing the model?



## 4. Add one explanation view and propose one mitigation
Generate a lightweight explanation artifact and write a short mitigation note grounded in the audit results.

**Checkpoint:** Checkpoint 4 verifies that the notebook produces at least one explanation artifact or ranked feature list and includes a short mitigation note tied to the subgroup audit, not just the global score.


In [ ]:
# Choose ONE explanation path based on what is available in the notebook environment.
# Do not skip directly here: only explain the tuned model AFTER you have checked
# its validation behavior and subgroup audit.

explanation_note = []

# Option A: global permutation importance
from sklearn.inspection import permutation_importance

result = permutation_importance(
    ...,
    ...,
    ...,
    n_repeats=5,
    random_state=7,
)

# TODO: print the top 5 features by importance

# Option B: if SHAP is available, explain a small sample
# import shap
# explainer = shap.Explainer(best_model.predict, X_sample_transformed)
# shap_values = explainer(X_sample_transformed)

# TODO: write 3-4 lines:
# 1. Which subgroup metric worried you most?
# 2. Which feature(s) seemed influential?
# 3. One mitigation you would test next (threshold, feature review, reweighting, or model change)
# 4. One reason this explanation should be treated cautiously



## Homework — HW6 — Fairness memo for a tuned model
Starting from your lab notebook, compare two candidate models or two threshold settings for the same recommendation task. Report overall recall plus at least two subgroup metrics, generate one explanation artifact, and write a brief memo recommending which version is safer to continue studying and why.

**Deliverable:** One runnable notebook plus a one-page memo. The memo must name the validation setup, the chosen metric, one observed subgroup gap, one limitation of the explanation method used, and one next mitigation to test. Do not give clinical treatment advice; stay at the model-evaluation level.


In [ ]:
# HW workspace

